In [58]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

In [47]:
train = pd.read_csv("dataset/train.csv")
test  = pd.read_csv("dataset/test.csv")

In [48]:
X_train = train.drop("Premium Amount", axis=1)
y_train = train["Premium Amount"]
X_test = test.copy()

In [49]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = [col for col in X_train.select_dtypes(include=['object', 'category']).columns 
                        if col != 'Policy Start Date']

categorical_ordinal = ['Education Level', 'Policy Type', 'Customer Feedback', 'Exercise Frequency']
categorical_nominal = [col for col in categorical_features if col not in categorical_ordinal]

In [54]:
# Phân tích chi tiết theo nhóm
print("=" * 90)
print("DETAILED MISSING VALUES ANALYSIS")
print("=" * 90)

# Xác định loại biến
def get_feature_type(col):
    if col in numeric_features:
        return 'Numerical'
    elif col in categorical_ordinal:
        return 'Cat-Ordinal'
    elif col in categorical_nominal:
        return 'Cat-Nominal'
    else:
        return 'Other'

# Tính toán
missing_info = pd.DataFrame({
    'Feature': X_train.columns,
    'Feature Type': [get_feature_type(col) for col in X_train.columns],
    'Missing Count': X_train.isnull().sum().values,
    'Missing %': (X_train.isnull().sum() / len(X_train) * 100).values,
    'Present Count': X_train.notnull().sum().values,
    'Data Type': X_train.dtypes.values
})

# Phân loại theo mức độ missing
missing_info['Missing Level'] = pd.cut(
    missing_info['Missing %'],
    bins=[-0.1, 0, 5, 15, 100],
    labels=['None', 'Low (0-5%)', 'Medium (5-15%)', 'High (>15%)']
)

# Chỉ hiển thị features có missing
missing_only = missing_info[missing_info['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("\nFEATURES WITH MISSING VALUES:\n")
print(f"{'Feature':<30} | {'Type':<15} | {'Missing':<12} | {'%':<8} | {'Level':<15}")
print("-" * 95)
for idx, row in missing_only.iterrows():
    print(f"{row['Feature']:<30} | {row['Feature Type']:<15} | {row['Missing Count']:>10,} | {row['Missing %']:>6.2f}% | {row['Missing Level']:<15}")

# Overall statistics
print("\n" + "=" * 90)
print("OVERALL STATISTICS:")
print(f"   • Total observations: {len(X_train):,}")
print(f"   • Total features: {len(X_train.columns)}")
print(f"   • Features with missing: {len(missing_only)}")
print(f"   • Features without missing: {len(X_train.columns) - len(missing_only)}")
print(f"   • Total missing cells: {X_train.isnull().sum().sum():,}")
print(f"   • Overall missing rate: {(X_train.isnull().sum().sum() / (len(X_train) * len(X_train.columns)) * 100):.2f}%")
print("=" * 90)

DETAILED MISSING VALUES ANALYSIS

FEATURES WITH MISSING VALUES:

Feature                        | Type            | Missing      | %        | Level          
-----------------------------------------------------------------------------------------------
Previous Claims                | Numerical       |    364,029 |  30.34% | High (>15%)    
Occupation                     | Cat-Nominal     |    358,075 |  29.84% | High (>15%)    
Credit Score                   | Numerical       |    137,882 |  11.49% | Medium (5-15%) 
Number of Dependents           | Numerical       |    109,672 |   9.14% | Medium (5-15%) 
Customer Feedback              | Cat-Ordinal     |     77,824 |   6.49% | Medium (5-15%) 
Health Score                   | Numerical       |     74,076 |   6.17% | Medium (5-15%) 
Annual Income                  | Numerical       |     44,949 |   3.75% | Low (0-5%)     
Age                            | Numerical       |     18,705 |   1.56% | Low (0-5%)     
Marital Status            

## Missing Values Processing Pipeline